# The rocket that exploded because a number did not fit

Ariane 5, 4 June 1996. A 64-bit floating-point value was converted to a 16-bit
integer, as it had been on Ariane 4 since 1988. Ariane 5 flew faster.

In [1]:
import numpy as np

ariane4_max = 30_000.0     # the largest horizontal-bias value the old rocket ever produced
ariane5_value = 36_000.0   # the new rocket, 37 seconds after launch

INT16_MAX = 2**15 - 1
print("16-bit signed range:", -2**15, "…", INT16_MAX)
print(ariane4_max <= INT16_MAX, ariane5_value <= INT16_MAX)

16-bit signed range: -32768 … 32767
True False


## What a conversion does when the value does not fit

Depending on the language and the settings it either **wraps silently** or
**raises**. NumPy shows both behaviors.

In [2]:
print("array conversion wraps silently:", np.array([ariane5_value]).astype(np.int16)[0])
try:
    np.int16(ariane5_value)
except OverflowError as e:
    print("scalar conversion raises:      ", e)

array conversion wraps silently: -29536
scalar conversion raises:       Python integer 36000 out of bounds for int16


On Ariane 5 the conversion raised, the exception was not handled, and the
inertial reference unit shut itself down. The backup unit, identical in hardware
and software, had failed the same way 50 milliseconds earlier.

## The fix is a check, and the lesson is where the assumption lives

In [3]:
def to_int16_checked(x):
    if not -2**15 <= x <= INT16_MAX:
        raise ValueError(f"{x} does not fit in 16 bits: input outside the assumed range")
    return int(x)

for v in (ariane4_max, ariane5_value):
    try:
        print(v, "->", to_int16_checked(v))
    except ValueError as e:
        print(v, "->", e)

30000.0 -> 30000
36000.0 -> 36000.0 does not fit in 16 bits: input outside the assumed range


The Ariane 4 engineers *had* analyzed the range, in a document rather than in
the code. When the inputs changed, nothing in the program knew.

**What we used:** integer types and their ranges, exceptions, and the practice of
putting assumptions in the code (session 4, sections 5, 11 and 12).